# Futures-Only Signal IC: Hedged ADR / Index Future Position

This notebook evaluates the futures-only signal at `13:00`, `13:30`, `14:00`, `14:30`, `15:00`, `15:30`, `15:50`, and `15:59` ET by measuring a hedged ADR / index-future position.

For each ticker and date, the realized return is defined as:

`hedged_return = stock_return - beta(date) * future_return`

- `beta(date)` comes from `data/processed/models/ordinary_betas_index_only.csv`, which is the same beta file used to create the futures-only signal.
- The future symbol comes from `adr_info.csv` and `futures_symbols.csv`, which is the same mapping used to create the futures-only signal.
- `ADR close` horizon: `stock_return` is ADR NBBO entry mid to ADR `PX_LAST` close, and `future_return` is entry future price to the future price at `16:00` ET.
- `Next ordinary open` horizon: `stock_return` is ADR NBBO entry mid to the next ordinary open, adjusted only for a next-day ordinary ex-date, and `future_return` is entry future price to the future price at the next ordinary market open.
- Only dates with a true next-calendar-day ordinary session are kept, so Fridays and ordinary-holiday eves drop automatically.
- The notebook also compares the magnitude of the hedged returns across the two horizons on overlapping observations.

The notebook is standalone and does not import repo code.


In [1]:
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandas_market_calendars as mcal
from tqdm.auto import tqdm

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)


In [2]:
DATA_DIR = Path('data') if Path('data/raw/adr_info.csv').exists() else Path('../data')
assert (DATA_DIR / 'raw' / 'adr_info.csv').exists(), 'Could not resolve data directory.'

SIGNAL_DIR = DATA_DIR / 'processed' / 'futures_only_signal'
ADR_NBBO_DIR = DATA_DIR / 'raw' / 'adrs' / 'bbo-1m' / 'nbbo'
ADR_CLOSE_FILE = DATA_DIR / 'raw' / 'adrs' / 'adr_PX_LAST_adjust_none.csv'
ORD_OPEN_NONE_FILE = DATA_DIR / 'processed' / 'ordinary' / 'ord_open_to_usd_adr_PX_OPEN_adjust_none.csv'
ORD_OPEN_ALL_FILE = DATA_DIR / 'processed' / 'ordinary' / 'ord_open_to_usd_adr_PX_OPEN_adjust_all.csv'
ADR_INFO_FILE = DATA_DIR / 'raw' / 'adr_info.csv'
FUTURES_SYMBOLS_FILE = DATA_DIR / 'raw' / 'futures_symbols.csv'
BETAS_FILE = DATA_DIR / 'processed' / 'models' / 'ordinary_betas_index_only.csv'
FUTURES_DIR = DATA_DIR / 'processed' / 'futures' / 'converted_minute_bars'

ENTRY_TIMES = ['13:00', '13:30', '14:00', '14:30', '15:00', '15:30', '15:50', '15:59']
ENTRY_LOOKBACK = pd.Timedelta('30min')
MIN_OBS_TICKER = 30
NEXT_OPEN_RATIO_ATOL = 1e-5
REGULAR_CLOSE_ONLY = True

adr_info = pd.read_csv(ADR_INFO_FILE)
adr_info['ticker'] = adr_info['adr'].str.replace(' US Equity', '', regex=False)
adr_info['index_future_bbg'] = adr_info['index_future_bbg'].astype(str).str.strip()
ord_to_adr = adr_info.set_index('id')['ticker'].to_dict()

futures_symbols = pd.read_csv(FUTURES_SYMBOLS_FILE)
futures_symbols['bloomberg_symbol'] = futures_symbols['bloomberg_symbol'].astype(str).str.strip()
futures_symbols['first_rate_symbol'] = futures_symbols['first_rate_symbol'].astype(str).str.strip()

ticker_meta = adr_info.merge(
    futures_symbols[['bloomberg_symbol', 'first_rate_symbol']],
    left_on='index_future_bbg',
    right_on='bloomberg_symbol',
    how='left',
)
ticker_meta['future_symbol'] = ticker_meta['first_rate_symbol'].astype(str).str.strip()
ticker_meta.loc[ticker_meta['future_symbol'] == '', 'future_symbol'] = np.nan
ticker_to_future = ticker_meta.set_index('ticker')['future_symbol'].to_dict()
ticker_to_exchange = ticker_meta.set_index('ticker')['exchange'].to_dict()

betas = pd.read_csv(BETAS_FILE, index_col=0, parse_dates=True).sort_index()

adr_close = pd.read_csv(ADR_CLOSE_FILE, index_col=0, parse_dates=True).sort_index()
adr_close.index = pd.DatetimeIndex(adr_close.index).normalize()

ord_open_none = (
    pd.read_csv(ORD_OPEN_NONE_FILE, index_col=0, parse_dates=True)
    .sort_index()
    .rename(columns=ord_to_adr)
)
ord_open_all = (
    pd.read_csv(ORD_OPEN_ALL_FILE, index_col=0, parse_dates=True)
    .sort_index()
    .rename(columns=ord_to_adr)
)

available_signal_tickers = sorted(
    p.name.split('=', 1)[1]
    for p in SIGNAL_DIR.glob('ticker=*')
    if p.is_dir()
)
available_nbbo_tickers = sorted(
    p.name.split('=', 1)[1]
    for p in ADR_NBBO_DIR.glob('ticker=*')
    if p.is_dir()
)
tickers = sorted(
    set(adr_info['ticker'])
    .intersection(available_signal_tickers)
    .intersection(available_nbbo_tickers)
)

ny_sched = mcal.get_calendar('NYSE').schedule(
    start_date=adr_close.index.min(),
    end_date=adr_close.index.max(),
)
ny_close = ny_sched['market_close'].dt.tz_convert('America/New_York')
regular_close_dates = pd.DatetimeIndex(
    ny_close[ny_close.dt.time == dt.time(16, 0)].index
).normalize()
regular_close_dates = regular_close_dates.drop(pd.Timestamp('2025-01-09'), errors='ignore')

open_times_by_exchange = {}
for exchange in sorted(ticker_meta['exchange'].dropna().unique()):
    ex_sched = mcal.get_calendar(exchange).schedule(
        start_date=ord_open_none.index.min(),
        end_date=ord_open_none.index.max(),
    )
    open_times_by_exchange[exchange] = ex_sched['market_open'].dt.tz_convert('America/New_York')

print(f'Tickers in evaluation universe: {len(tickers)}')
print(f'Beta date range: {betas.index.min().date()} -> {betas.index.max().date()}')
print(f'Regular NYSE close dates: {len(regular_close_dates):,}')


Tickers in evaluation universe: 76
Beta date range: 2018-08-23 -> 2026-03-06
Regular NYSE close dates: 1,955


In [3]:
def _normalize_dates(index):
    idx = pd.DatetimeIndex(index)
    if idx.tz is not None:
        idx = idx.tz_convert('America/New_York').tz_localize(None)
    return idx.normalize()


def _coerce_ny_ns(index_like):
    idx = pd.DatetimeIndex(index_like)
    if idx.tz is None:
        idx = idx.tz_localize('America/New_York')
    else:
        idx = idx.tz_convert('America/New_York')
    return pd.DatetimeIndex(idx.asi8, tz='America/New_York')


def _safe_corr(x: pd.Series, y: pd.Series, min_obs: int = 2) -> float:
    z = pd.concat([x, y], axis=1).dropna()
    if len(z) < min_obs:
        return np.nan
    if z.iloc[:, 0].std() == 0 or z.iloc[:, 1].std() == 0:
        return np.nan
    return float(z.iloc[:, 0].corr(z.iloc[:, 1]))


def _last_signal_at_or_before(signal_df: pd.DataFrame, entry_times):
    signal = signal_df['signal'].astype(float)
    out = {}
    for entry_time in entry_times:
        s = signal.between_time('00:00', entry_time)
        out[entry_time] = pd.Series(
            s.to_numpy(dtype=float),
            index=_normalize_dates(s.index),
        ).groupby(level=0).last().sort_index()
    return out


def _last_series_at_or_before(series: pd.Series, entry_times, lookback: pd.Timedelta):
    out = {}
    for entry_time in entry_times:
        start_time = (pd.Timestamp(entry_time) - lookback).strftime('%H:%M')
        s = series.between_time(start_time, entry_time)
        out[entry_time] = pd.Series(
            s.to_numpy(dtype=float),
            index=_normalize_dates(s.index),
        ).groupby(level=0).last().sort_index()
    return out


def _load_futures_close_series(path: Path) -> pd.Series:
    fut_df = pd.read_parquet(path, columns=['timestamp', 'close']).dropna(subset=['close'])
    fut_df['timestamp'] = pd.to_datetime(fut_df['timestamp'])
    fut_df = fut_df.sort_values('timestamp').set_index('timestamp')
    fut_df.index = _coerce_ny_ns(fut_df.index)
    return fut_df['close'].astype(float)


def _next_open_targets(
    open_none: pd.Series,
    open_all: pd.Series,
    open_times: pd.Series,
    ratio_atol: float = 1e-5,
):
    df = pd.concat({'open_none': open_none, 'open_all': open_all}, axis=1).dropna().sort_index()
    if df.empty:
        return pd.DataFrame(columns=['next_open_adj', 'exit_multiplier', 'next_open_time'])

    df['cum_adj'] = df['open_all'] / df['open_none']
    next_date = df.index.to_series().shift(-1)
    valid = next_date.eq(df.index.to_series() + pd.Timedelta(days=1))
    out = df.loc[valid, ['open_none', 'cum_adj']].copy()
    out['next_date'] = next_date.loc[out.index]
    out['next_open_none'] = df['open_none'].shift(-1).loc[out.index]
    out['next_cum_adj'] = df['cum_adj'].shift(-1).loc[out.index]

    ratio = out['next_cum_adj'] / out['cum_adj']
    ratio = ratio.where(
        ~np.isclose(ratio, 1.0, atol=ratio_atol, rtol=0.0),
        1.0,
    )
    out['exit_multiplier'] = ratio
    out['next_open_adj'] = out['next_open_none'] * out['exit_multiplier']

    next_open_time = open_times.reindex(pd.DatetimeIndex(out['next_date']))
    out['next_open_time'] = pd.Series(_coerce_ny_ns(next_open_time.to_numpy()), index=out.index)
    return out[['next_open_adj', 'exit_multiplier', 'next_open_time']]


def _series_at_target_timestamps(series: pd.Series, target_times: pd.Series, tolerance: pd.Timedelta):
    if len(series) == 0 or len(target_times) == 0:
        return pd.Series(dtype=float)

    target_df = target_times.dropna().rename('target_time').sort_values().reset_index()
    if target_df.empty:
        return pd.Series(dtype=float)
    target_df.columns = ['date', 'target_time']
    target_df['target_time'] = _coerce_ny_ns(target_df['target_time'])

    price_df = series.sort_index().rename('price').reset_index()
    price_df.columns = ['timestamp', 'price']
    price_df['timestamp'] = _coerce_ny_ns(price_df['timestamp'])
    target_df = target_df.sort_values('target_time').reset_index(drop=True)
    price_df = price_df.sort_values('timestamp').reset_index(drop=True)

    merged = pd.merge_asof(
        target_df,
        price_df,
        left_on='target_time',
        right_on='timestamp',
        direction='backward',
        tolerance=tolerance,
    )
    out = merged.dropna(subset=['price']).set_index('date')['price'].astype(float).sort_index()
    out.index = pd.DatetimeIndex(out.index).normalize()
    return out[~out.index.duplicated(keep='last')]


def _build_ic_outputs(
    panel_df: pd.DataFrame,
    signal_col: str = 'signal',
    ret_col: str = 'hedged_return',
    min_obs_ticker: int = 30,
):
    ticker_rows = []
    for (entry_time, ticker), g in panel_df.groupby(['entry_time', 'ticker']):
        ticker_rows.append(
            {
                'entry_time': entry_time,
                'ticker': ticker,
                'n_obs': len(g),
                'ic': _safe_corr(g[signal_col], g[ret_col], min_obs=min_obs_ticker),
            }
        )

    ticker_ic = pd.DataFrame(ticker_rows)
    if ticker_ic.empty:
        empty = pd.DataFrame(
            columns=['mean_per_ticker_ic', 'median_ticker_ic', 'n_tickers', 'total_obs', 'pooled_ic', 'pooled_obs']
        )
        return ticker_ic, empty

    summary = (
        ticker_ic.groupby('entry_time')
        .agg(
            mean_per_ticker_ic=('ic', 'mean'),
            median_ticker_ic=('ic', 'median'),
            n_tickers=('ic', lambda s: int(s.notna().sum())),
            total_obs=('n_obs', 'sum'),
        )
        .sort_index()
    )

    pooled_rows = []
    for entry_time, g in panel_df.groupby('entry_time'):
        pooled_rows.append(
            {
                'entry_time': entry_time,
                'pooled_ic': _safe_corr(g[signal_col], g[ret_col], min_obs=min_obs_ticker),
                'pooled_obs': len(g),
            }
        )

    pooled_df = pd.DataFrame(pooled_rows).set_index('entry_time').sort_index()
    summary = summary.join(pooled_df)
    return ticker_ic.sort_values(['entry_time', 'ticker']), summary


In [4]:
close_rows = []
next_open_rows = []
diagnostic_rows = []

for ticker in tqdm(tickers, desc='Building hedged evaluation panels'):
    future_symbol = ticker_to_future.get(ticker)
    if not isinstance(future_symbol, str) or future_symbol == 'nan' or future_symbol == '':
        continue

    future_path = FUTURES_DIR / f'symbol={future_symbol}' / f'{future_symbol}_close_to_usd_1min.parquet'
    if not future_path.exists():
        continue

    signal_path = SIGNAL_DIR / f'ticker={ticker}' / 'data.parquet'
    nbbo_path = ADR_NBBO_DIR / f'ticker={ticker}' / 'data.parquet'

    signal_df = pd.read_parquet(signal_path, columns=['signal'])
    nbbo_df = pd.read_parquet(nbbo_path, columns=['nbbo_bid', 'nbbo_ask'])
    future_close = _load_futures_close_series(future_path)

    signal_by_time = _last_signal_at_or_before(signal_df, ENTRY_TIMES)
    adr_entry_mid = _last_series_at_or_before(
        ((nbbo_df['nbbo_bid'] + nbbo_df['nbbo_ask']) / 2.0).astype(float),
        ENTRY_TIMES,
        ENTRY_LOOKBACK,
    )
    future_entry = _last_series_at_or_before(future_close, ENTRY_TIMES, ENTRY_LOOKBACK)
    future_close_exit = _last_series_at_or_before(future_close, ['16:00'], ENTRY_LOOKBACK)['16:00']

    beta_s = betas[ticker].dropna().astype(float) if ticker in betas.columns else pd.Series(dtype=float)
    close_exit = adr_close[ticker].dropna().astype(float) if ticker in adr_close.columns else pd.Series(dtype=float)

    exchange = ticker_to_exchange.get(ticker)
    open_times = open_times_by_exchange.get(exchange, pd.Series(dtype='datetime64[ns, America/New_York]'))
    if ticker in ord_open_none.columns and ticker in ord_open_all.columns and len(open_times) > 0:
        next_open_targets = _next_open_targets(
            ord_open_none[ticker].dropna().astype(float),
            ord_open_all[ticker].dropna().astype(float),
            open_times,
            ratio_atol=NEXT_OPEN_RATIO_ATOL,
        )
        future_next_open_exit = _series_at_target_timestamps(
            future_close,
            next_open_targets['next_open_time'],
            ENTRY_LOOKBACK,
        )
    else:
        next_open_targets = pd.DataFrame(columns=['next_open_adj', 'exit_multiplier', 'next_open_time'])
        future_next_open_exit = pd.Series(dtype=float)

    diagnostic_rows.append(
        {
            'ticker': ticker,
            'future_symbol': future_symbol,
            'close_obs': len(close_exit),
            'next_open_obs': len(next_open_targets),
            'adjusted_next_open_days': int(
                (~np.isclose(next_open_targets.get('exit_multiplier', pd.Series(dtype=float)), 1.0, atol=NEXT_OPEN_RATIO_ATOL, rtol=0.0)).sum()
            ) if len(next_open_targets) else 0,
        }
    )

    for entry_time in ENTRY_TIMES:
        signal_s = signal_by_time.get(entry_time, pd.Series(dtype=float))
        adr_entry_s = adr_entry_mid.get(entry_time, pd.Series(dtype=float))
        fut_entry_s = future_entry.get(entry_time, pd.Series(dtype=float))

        if REGULAR_CLOSE_ONLY:
            signal_s = signal_s[signal_s.index.isin(regular_close_dates)]
            adr_entry_s = adr_entry_s[adr_entry_s.index.isin(regular_close_dates)]
            fut_entry_s = fut_entry_s[fut_entry_s.index.isin(regular_close_dates)]

        common_close = (
            signal_s.index
            .intersection(adr_entry_s.index)
            .intersection(fut_entry_s.index)
            .intersection(close_exit.index)
            .intersection(future_close_exit.index)
            .intersection(beta_s.index)
        )
        if len(common_close) > 0:
            adr_ret = close_exit.loc[common_close].astype(float) / adr_entry_s.loc[common_close].astype(float) - 1.0
            fut_ret = future_close_exit.loc[common_close].astype(float) / fut_entry_s.loc[common_close].astype(float) - 1.0
            hedged_ret = adr_ret - beta_s.loc[common_close].astype(float) * fut_ret
            close_rows.append(
                pd.DataFrame(
                    {
                        'date': common_close,
                        'ticker': ticker,
                        'entry_time': entry_time,
                        'signal': signal_s.loc[common_close].to_numpy(dtype=float),
                        'beta': beta_s.loc[common_close].to_numpy(dtype=float),
                        'stock_return': adr_ret.to_numpy(dtype=float),
                        'future_return': fut_ret.to_numpy(dtype=float),
                        'hedged_return': hedged_ret.to_numpy(dtype=float),
                    }
                )
            )

        common_next = (
            signal_s.index
            .intersection(adr_entry_s.index)
            .intersection(fut_entry_s.index)
            .intersection(next_open_targets.index)
            .intersection(future_next_open_exit.index)
            .intersection(beta_s.index)
        )
        if len(common_next) > 0:
            stock_ret = (
                next_open_targets.loc[common_next, 'next_open_adj'].astype(float)
                / adr_entry_s.loc[common_next].astype(float)
                - 1.0
            )
            fut_ret = future_next_open_exit.loc[common_next].astype(float) / fut_entry_s.loc[common_next].astype(float) - 1.0
            hedged_ret = stock_ret - beta_s.loc[common_next].astype(float) * fut_ret
            next_open_rows.append(
                pd.DataFrame(
                    {
                        'date': common_next,
                        'ticker': ticker,
                        'entry_time': entry_time,
                        'signal': signal_s.loc[common_next].to_numpy(dtype=float),
                        'beta': beta_s.loc[common_next].to_numpy(dtype=float),
                        'stock_return': stock_ret.to_numpy(dtype=float),
                        'future_return': fut_ret.to_numpy(dtype=float),
                        'exit_multiplier': next_open_targets.loc[common_next, 'exit_multiplier'].to_numpy(dtype=float),
                        'hedged_return': hedged_ret.to_numpy(dtype=float),
                    }
                )
            )

close_panel = pd.concat(close_rows, ignore_index=True) if close_rows else pd.DataFrame()
next_open_panel = pd.concat(next_open_rows, ignore_index=True) if next_open_rows else pd.DataFrame()
diagnostics = pd.DataFrame(diagnostic_rows).sort_values('ticker').reset_index(drop=True)

print(f'Close-horizon rows: {len(close_panel):,}')
print(f'Next-open rows: {len(next_open_panel):,}')
print(f'Next-open corporate-action-adjusted rows: {int((~np.isclose(next_open_panel.get("exit_multiplier", pd.Series(dtype=float)), 1.0, atol=NEXT_OPEN_RATIO_ATOL, rtol=0.0)).sum() if len(next_open_panel) else 0):,}')
display(diagnostics.head())


Building hedged evaluation panels:   0%|          | 0/76 [00:00<?, ?it/s]

Close-horizon rows: 0
Next-open rows: 0
Next-open corporate-action-adjusted rows: 0


,ticker,future_symbol,close_obs,next_open_obs,adjusted_next_open_days
0,AEG,FESX,1973,1474,584
1,ARGX,FESX,1973,1466,0
2,ASML,FESX,1973,1474,13
3,AZN,FTUK,1647,1458,15
4,BABA,MME,1973,0,0


In [5]:
close_ticker_ic, close_summary = _build_ic_outputs(
    close_panel,
    signal_col='signal',
    ret_col='hedged_return',
    min_obs_ticker=MIN_OBS_TICKER,
)

print('Hedged ADR close horizon')
print('Primary metric: mean per-ticker IC by entry time')
display(close_summary)

close_ic_matrix = close_ticker_ic.pivot(index='ticker', columns='entry_time', values='ic')
display(close_ic_matrix.sort_index())


KeyError: 'entry_time'

In [ ]:
next_open_ticker_ic, next_open_summary = _build_ic_outputs(
    next_open_panel,
    signal_col='signal',
    ret_col='hedged_return',
    min_obs_ticker=MIN_OBS_TICKER,
)

print('Hedged next ordinary open horizon')
print('Primary metric: mean per-ticker IC by entry time')
display(next_open_summary)

next_open_ic_matrix = next_open_ticker_ic.pivot(index='ticker', columns='entry_time', values='ic')
display(next_open_ic_matrix.sort_index())


In [ ]:
combined_summary = pd.concat(
    {
        'hedged_adr_close': close_summary,
        'hedged_next_ord_open': next_open_summary,
    },
    names=['horizon', 'entry_time'],
)
display(combined_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

close_summary['mean_per_ticker_ic'].plot(ax=axes[0], marker='o', label='Hedged ADR close')
next_open_summary['mean_per_ticker_ic'].plot(ax=axes[0], marker='o', label='Hedged next ordinary open')
axes[0].axhline(0.0, color='gray', linewidth=0.8)
axes[0].set_title('Mean Per-Ticker IC')
axes[0].set_xlabel('Entry Time')
axes[0].set_ylabel('IC')
axes[0].legend()

close_summary['pooled_ic'].plot(ax=axes[1], marker='o', label='Hedged ADR close')
next_open_summary['pooled_ic'].plot(ax=axes[1], marker='o', label='Hedged next ordinary open')
axes[1].axhline(0.0, color='gray', linewidth=0.8)
axes[1].set_title('Pooled IC')
axes[1].set_xlabel('Entry Time')
axes[1].set_ylabel('IC')
axes[1].legend()

plt.tight_layout()

adjusted_rows = next_open_panel.loc[
    ~np.isclose(next_open_panel['exit_multiplier'], 1.0, atol=NEXT_OPEN_RATIO_ATOL, rtol=0.0),
    ['date', 'ticker', 'entry_time', 'exit_multiplier'],
].sort_values(['date', 'ticker', 'entry_time'])

print(f'Rows with a next-day ordinary corporate-action adjustment: {len(adjusted_rows):,}')
display(adjusted_rows.head(20))
display(diagnostics.sort_values('adjusted_next_open_days', ascending=False).head(20))


In [ ]:
magnitude_comparison = (
    close_panel[['date', 'ticker', 'entry_time', 'hedged_return']]
    .rename(columns={'hedged_return': 'close_hedged_return'})
    .merge(
        next_open_panel[['date', 'ticker', 'entry_time', 'hedged_return']].rename(
            columns={'hedged_return': 'next_open_hedged_return'}
        ),
        on=['date', 'ticker', 'entry_time'],
        how='inner',
    )
)

magnitude_comparison['abs_close_hedged_return'] = magnitude_comparison['close_hedged_return'].abs()
magnitude_comparison['abs_next_open_hedged_return'] = magnitude_comparison['next_open_hedged_return'].abs()
magnitude_comparison['abs_diff'] = (
    magnitude_comparison['abs_next_open_hedged_return'] - magnitude_comparison['abs_close_hedged_return']
)
magnitude_comparison['abs_ratio'] = (
    magnitude_comparison['abs_next_open_hedged_return']
    / magnitude_comparison['abs_close_hedged_return'].replace(0.0, np.nan)
).replace([np.inf, -np.inf], np.nan)
magnitude_comparison['next_open_larger'] = magnitude_comparison['abs_diff'] > 0

magnitude_summary = (
    magnitude_comparison.groupby('entry_time')
    .agg(
        mean_abs_close_hedged_return=('abs_close_hedged_return', 'mean'),
        mean_abs_next_open_hedged_return=('abs_next_open_hedged_return', 'mean'),
        median_abs_close_hedged_return=('abs_close_hedged_return', 'median'),
        median_abs_next_open_hedged_return=('abs_next_open_hedged_return', 'median'),
        mean_abs_diff=('abs_diff', 'mean'),
        median_abs_diff=('abs_diff', 'median'),
        mean_abs_ratio=('abs_ratio', 'mean'),
        median_abs_ratio=('abs_ratio', 'median'),
        pct_next_open_larger=('next_open_larger', 'mean'),
        n_obs=('abs_diff', 'size'),
    )
    .sort_index()
)

magnitude_by_ticker = (
    magnitude_comparison.groupby(['ticker', 'entry_time'])
    .agg(
        mean_abs_close_hedged_return=('abs_close_hedged_return', 'mean'),
        mean_abs_next_open_hedged_return=('abs_next_open_hedged_return', 'mean'),
        mean_abs_diff=('abs_diff', 'mean'),
        median_abs_ratio=('abs_ratio', 'median'),
        pct_next_open_larger=('next_open_larger', 'mean'),
        n_obs=('abs_diff', 'size'),
    )
    .reset_index()
    .sort_values(['entry_time', 'mean_abs_diff'], ascending=[True, False])
)

print('Hedged return magnitude comparison on overlapping observations')
display(magnitude_summary)
display(magnitude_by_ticker.head(20))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

magnitude_summary[['mean_abs_close_hedged_return', 'mean_abs_next_open_hedged_return']].plot(
    ax=axes[0],
    marker='o',
)
axes[0].set_title('Mean Absolute Hedged Return Magnitude')
axes[0].set_xlabel('Entry Time')
axes[0].set_ylabel('Absolute Hedged Return')

(100.0 * magnitude_summary['pct_next_open_larger']).plot(ax=axes[1], marker='o', color='black')
axes[1].axhline(50.0, color='gray', linewidth=0.8)
axes[1].set_title('Pct Next Open Magnitude > Close Magnitude')
axes[1].set_xlabel('Entry Time')
axes[1].set_ylabel('Percent of Overlapping Rows')

plt.tight_layout()


In [ ]:
OUT_DIR = DATA_DIR / 'processed' / 'reports' / 'signal_ic' / 'futures_only_hedged_close_vs_next_ord_open'
OUT_DIR.mkdir(parents=True, exist_ok=True)

close_summary.to_csv(OUT_DIR / 'close_summary.csv')
next_open_summary.to_csv(OUT_DIR / 'next_open_summary.csv')
close_ticker_ic.to_csv(OUT_DIR / 'close_ticker_ic.csv', index=False)
next_open_ticker_ic.to_csv(OUT_DIR / 'next_open_ticker_ic.csv', index=False)
combined_summary.to_csv(OUT_DIR / 'combined_summary.csv')
magnitude_summary.to_csv(OUT_DIR / 'magnitude_summary.csv')
magnitude_by_ticker.to_csv(OUT_DIR / 'magnitude_by_ticker.csv', index=False)
diagnostics.to_csv(OUT_DIR / 'ticker_diagnostics.csv', index=False)

print(f'Saved outputs to: {OUT_DIR}')
